# 中国八城市课程数据包：质量审计

## tl;dr

本Notebook是教材数据发布前的可执行审计记录。它检查八个城市的数据文件是否齐全，
GeoJSON与CSV主键能否一一连接，GeoTIFF是否保留空间参考，元数据计数与校验值是否一致。
`critical`或`high`级问题会阻止发布。

## Context & Methods

数据包服务于第6—16章的课堂练习。每城采用相同的数据契约，但研究范围是城市中心附近的
小尺度教学裁剪。检查脚本只验证文件结构、连接关系与发布完整性；它无法证明OpenStreetMap
在不同城市中的设施覆盖程度完全一致，因此跨城市比较时必须把平台覆盖差异列为限制。

In [1]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "scripts").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data" / "china_city_cases"
subprocess.run(
    [sys.executable, str(ROOT / "scripts" / "validate_china_city_cases.py")],
    cwd=ROOT,
    check=True,
)

{
  "status": "PASS",
  "publishable": true,
  "severity_counts": {
    "critical": 0,
    "high": 0,
    "medium": 0,
    "low": 0
  },
  "check_count": 108
}
质量报告：/home/runner/work/urban-spatial-data-science/urban-spatial-data-science/data/china_city_cases/quality_report.json


CompletedProcess(args=['/opt/hostedtoolcache/Python/3.12.14/x64/bin/python', '/home/runner/work/urban-spatial-data-science/urban-spatial-data-science/scripts/validate_china_city_cases.py'], returncode=0)

## Data

In [2]:
catalog = pd.read_csv(DATA / "catalog.csv")
profiles = pd.read_csv(DATA / "city_profiles.csv")
grids = pd.read_csv(DATA / "all_grid_indicators.csv")
inventory = pd.read_csv(DATA / "data_inventory.csv")
report = json.loads((DATA / "quality_report.json").read_text(encoding="utf-8"))

catalog[["city_name_zh", "facility_count", "network_edge_count", "grid_count", "extract_date"]]

,city_name_zh,facility_count,network_edge_count,grid_count,extract_date
0,北京,135,10052,64,2026-08-24
1,上海,265,14252,64,2026-08-24
2,南京,157,3777,64,2026-08-24
3,广州,227,14228,64,2026-08-24
4,成都,66,5723,64,2026-08-24
5,武汉,59,6473,64,2026-08-24
6,西安,92,3217,64,2026-08-24
7,杭州,164,7608,64,2026-08-24


## Results

In [3]:
audit_summary = pd.DataFrame([
    {
        "status": report["status"],
        "publishable": report["publishable"],
        "checks": report["check_count"],
        **report["severity_counts"],
    }
])
audit_summary

,status,publishable,checks,critical,high,medium,low
0,PASS,True,108,0,0,0,0


In [4]:
coverage = (
    grids.groupby(["city_id", "city_name_zh"], as_index=False)
    .agg(
        grid_cells=("cell_id", "nunique"),
        facilities=("facility_count", "sum"),
        road_length_km=("road_length_m", lambda values: values.sum() / 1000),
        elevation_min_m=("elevation_m", "min"),
        elevation_max_m=("elevation_m", "max"),
    )
)
coverage

,city_id,city_name_zh,grid_cells,facilities,road_length_km,elevation_min_m,elevation_max_m
0,beijing,北京,64,135,336.10854,42,71
1,chengdu,成都,64,66,355.74260,474,502
2,guangzhou,广州,64,227,357.01498,1,46
3,hangzhou,杭州,64,164,345.05484,6,82
4,nanjing,南京,64,157,202.27409,7,143
5,shanghai,上海,64,265,470.95942,-2,38
6,wuhan,武汉,64,59,309.65391,14,48
7,xian,西安,64,92,254.80101,370,395


In [5]:
# 关键一致性断言：失败时Notebook执行会停止。
assert report["publishable"] is True
assert set(catalog["city_id"]) == set(profiles["city_id"])
assert len(catalog) == 8
assert len(grids) == 8 * 64
assert grids["cell_id"].is_unique
assert inventory.groupby("city_id")["dataset_id"].nunique().eq(3).all()
print("全部发布门槛均已通过。")

全部发布门槛均已通过。


## Takeaways

- 八个城市均具备矢量、栅格和统计表三类课堂数据。
- 每城64个分析网格可由`cell_id`无损连接到指标表。
- GeoTIFF保留WGS84空间参考与无数据值；道路、设施和高程文件均记录来源。
- OSM覆盖差异、中心城区裁剪和教学推导指标仍是实质性限制，使用者应在研究报告中说明。